In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from mylibs import *
import os
import pandas as pd 

In [2]:
# Collect 520 products from pages 1 to 13
list_links = []

driver = webdriver.Chrome()
driver.maximize_window()

for page in range(1, 14):
    url = f'https://hasaki.vn/danh-muc/cham-soc-da-mat-c4.html?p={page}'
    driver.get(url)

    # Locate the product grid
    product_container = driver.find_element(by=By.XPATH, value='//div[@class="ProductGrid__grid width_common"]')


    # Find the product detail links
    tag_links = product_container.find_elements(by=By.CSS_SELECTOR, value='.v3_thumb_common_sp.relative')
    for tag_link in tag_links:
        link = tag_link.get_attribute('href') + '\n'
        list_links.append(link)
print(f'Tìm thấy {len(list_links)} link sản phẩm')
write_txt(f'facial_skin_care_product_links.txt', list_links)

Tìm thấy 520 link sản phẩm


True

### Collect customer reviews for each of the 520 products

In [3]:
# list_customers = []
# list_comments = []
# list_dates = []
# list_hours = []
# list_stars = []
# list_item_codes = []


# Read the links and extract data from each product page
list_links = read_txt('facial_skin_care_product_links.txt')

for i in range(0, len(list_links)):
    link = list_links[i].rstrip('\n')
    driver.get(link)

    
    # 1. Reviewer name
    tag_customers = driver.find_elements(by=By.XPATH, value='//strong[@class="txt_color_1"]')
    for tag_customer in tag_customers:
        customers = tag_customer.text
        # list_customers.append(customers)
    


    # 2. Review text
    tag_comments = driver.find_elements(by=By.XPATH, value='//div[@class="content_comment"]')
    for tag_comment in tag_comments:
        comments = tag_comment.text
        # list_comments.append(comments)
    
    


    # 3. Review date
    tag_dates = driver.find_elements(By.XPATH, value='//div[@class="timer_comment txt_666 space_bottom_5 right"]')
    for tag_date in tag_dates:
        date = tag_date.text.split("|")[-1].strip()
        # list_dates.append(date)
    


    # 4. Review time
    tag_hours = driver.find_elements(By.XPATH, value='//div[@class="timer_comment txt_666 space_bottom_5 right"]')
    for tag_hour in tag_hours:
        hour = tag_hour.text.split("|")[0].strip()
        # list_hours.append(hour)
    


    # 5. Star rating
    
    tag_stars = driver.find_elements(By.XPATH, value='//div[@class="number_start"]')
    star = 0
    for tag_star in tag_stars:
        data_star = tag_star.get_attribute('style')
        if 'width:' in data_star:  
            width_per = int(data_star.split("width:")[1].split("%")[0].strip())
            if width_per == 100:
                star = 5
            elif width_per >= 80:
                star = 4
            elif width_per >= 60:
                star = 3
            elif width_per >= 40:
                star = 2
            elif width_per>= 20:
                star = 1
            else:
                star = 0
        else:
            star = 0     # No width is specified in data_star
        # list_stars.append(star)
    


    # 6. Product code
    
    tag_item_code = driver.find_element(By.XPATH, value='//span[@class="item-sku txt_color_1"]')
    item_code = tag_item_code.text.replace('Mã sản phẩm: ', '')
    # list_item_codes.append(item_code)


    # Save the record as JSON
    data_json_dir = f'data/customer_reviews'
    if not os.path.exists(data_json_dir):
        os.makedirs(data_json_dir)

    no = i + 1
    file_name = f'{str(no).zfill(3)}.json'
    full_path = f'{data_json_dir}/{file_name}'
    data = {
        'Ho_ten_nguoi_mua': customers,
        'Noi_dung_binh_luan': comments,
        'Ngay_binh_luan': date,
        'Gio_binh_luan': hour,
        'So_sao': star,
        'Ma_san_pham': item_code
        
    }

    write_json(full_path, data)
    print(f'>>> Đã hoàn tất sản phẩm thứ {no}')
    print('-' * 50)

    
driver.close()



# print(len(list_customers), list_customers)
# print(len(list_comments), list_comments)
# print(len(list_dates), list_dates)
# print(len(list_hours), list_hours)
# print(len(list_stars), list_stars)
# print(len(list_item_codes), list_item_codes)


>>> Đã hoàn tất sản phẩm thứ 1
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 2
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 3
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 4
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 5
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 6
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 7
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 8
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 9
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 10
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 11
--------------------------------------------------
>>> Đã hoàn tất sản phẩm thứ 12
--------------------------------------------------
>>> Đã hoàn t

In [8]:
# Combine the JSON records into a CSV file
list_customers = []
list_comments = []
list_dates = []
list_hours = []
list_stars = []
list_item_codes = []



data_dir = f'data/customer_reviews'
for file_name in os.listdir(data_dir):
    if file_name.endswith('.json'):
        path = f'{data_dir}/{file_name}'
        dict_product = read_json(path)
        print(dict_product)

        # Append field values to their lists
        list_customers.append(dict_product['Ho_ten_nguoi_mua'])
        list_comments.append(dict_product['Noi_dung_binh_luan'])
        list_dates.append(dict_product['Ngay_binh_luan'])
        list_hours.append(dict_product['Gio_binh_luan'])
        list_stars.append(dict_product['So_sao'])
        list_item_codes.append(dict_product['Ma_san_pham'])
else:

    # Export the data to CSV
    df_product = pd.DataFrame({
        
        'Ho_ten_nguoi_mua': list_customers ,
        'Noi_dung_binh_luan': list_comments,
        'Ngay_binh_luan': list_dates,
        'Gio_binh_luan': list_hours,
        'So_sao': list_stars,
        'Ma_san_pham': list_item_codes
    })

    df_product.to_csv(f'customer_reviews.csv', index=False)


    

{'Ho_ten_nguoi_mua': 'ANH THƯ', 'Noi_dung_binh_luan': 'mình dùng 3 chai ròi, thấy thích hơn loại không mùi, cũng thấy đỡ bị mụn viêm hơn, còn cấp ẩm thì tương tự loại không mùi. MÌnh da nhạy cảm dễ bị kích ứng mà xài thích loại này, 10đ khong có nhưng nha', 'Ngay_binh_luan': '30/09/2024', 'Gio_binh_luan': '13: 36', 'So_sao': 5, 'Ma_san_pham': '318900012'}
{'Ho_ten_nguoi_mua': 'Diệu Trang Cao', 'Noi_dung_binh_luan': 'Trước giờ luôn xài sữa rửa mặt Cerave vì đủ sạch không quá sạch sâu nhưng mà rửa xong da rất mềm mịn mướt không bị khô căng nói chung rất thích rất ưng chắc là cả đời chỉ có Cerave là chân ái đối với mình thôi', 'Ngay_binh_luan': '14/10/2024', 'Gio_binh_luan': '00: 14', 'So_sao': 5, 'Ma_san_pham': '422208973'}
{'Ho_ten_nguoi_mua': 'Kim Ngân', 'Noi_dung_binh_luan': 'tẩy trang tốt nhất của loreal từng dùng,không bị rát da và giảm ma sát trên da khi tẩy trang,và cũng rất sạch nữa,không bị khô da... rất đáng tiền', 'Ngay_binh_luan': '23/06/2024', 'Gio_binh_luan': '11: 25', 'So_